# Respiratory lung tracking training and test workflow

Notebook preuzima ZIP arhivu koja sadrži 392 image-mask parova, trenira U-Net, prikazuje minimizaciju greške i zatim testira model na ručno uploadovanoj respiratornoj sekvenci.


In [ ]:
# 1) Unesi Google Drive link ka colab_training_dataset.zip arhivi.
# Arhiva mora sadržati folder images/ i folder masks/ sa ukupno 392 PNG parova.
DATASET_URL = 'https://drive.google.com/file/d/1k_YN_Py-Mum8ARfs_QkBiOvrkx6Pd49v/view?usp=sharing'

EPOCHS = 30
BATCH_SIZE = 16
LEARNING_RATE = 2e-4
SEED = 42
EXPECTED_PAIRS = 392

if DATASET_URL.startswith('PASTE_'):
    raise ValueError('U DATASET_URL nalepi Google Drive share link ka colab_training_dataset.zip arhivi.')


In [ ]:
# 2) Instalacija, preuzimanje i raspakivanje identičnog lokalnog trening skupa.
!pip -q install opencv-python-headless scipy pandas matplotlib gdown

import json, random, shutil, subprocess, tarfile, urllib.request, zipfile
from pathlib import Path
import cv2
import gdown
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from scipy import ndimage
from scipy.signal import savgol_filter
from tqdm.auto import tqdm
from IPython.display import display, Video

WORKDIR = Path('/content/respiratory_lung_tracking_train')
RAW_DIR = WORKDIR / 'raw'
PREPARED_DIR = WORKDIR / 'prepared'
IMAGE_DIR = PREPARED_DIR / 'images'
MASK_DIR = PREPARED_DIR / 'masks'
OUTPUT_DIR = WORKDIR / 'outputs'
MODEL_PATH = WORKDIR / 'best_lung_unet.pt'
for folder in (WORKDIR, RAW_DIR, PREPARED_DIR, IMAGE_DIR, MASK_DIR, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

archive_path = RAW_DIR / 'colab_training_dataset.zip'
result = gdown.download(url=DATASET_URL, output=str(archive_path), fuzzy=True, quiet=False)
if result is None:
    raise RuntimeError('Google Drive download nije uspeo. Proveri da li je ZIP arhiva podešena na Anyone with the link.')
extract_dir = RAW_DIR / 'dataset'
if zipfile.is_zipfile(archive_path):
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(extract_dir)
elif tarfile.is_tarfile(archive_path):
    with tarfile.open(archive_path, 'r:*') as archive:
        archive.extractall(extract_dir, filter='data')
else:
    raise RuntimeError('Preuzet fajl nije ZIP/TAR arhiva. Uploaduj colab_training_dataset.zip na Google Drive, ne folder link.')
print('Downloaded and extracted local V7 training dataset.')


In [ ]:
# 3) Provera strukture i priprema standardnih images/ + masks/ foldera.
IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

for folder in (IMAGE_DIR, MASK_DIR):
    for path in folder.glob('*.png'):
        path.unlink()

def write_pair(output_id, image_path, mask_path):
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if image is None or mask is None:
        return False
    if image.shape != mask.shape:
        mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
    mask = ((mask > 127).astype(np.uint8)) * 255
    if cv2.countNonZero(mask) < 500:
        return False
    cv2.imwrite(str(IMAGE_DIR / f'{output_id}.png'), image)
    cv2.imwrite(str(MASK_DIR / f'{output_id}.png'), mask)
    return True

def locate_pair_folders(root):
    candidates = []
    for image_dir in root.rglob('images'):
        mask_dir = image_dir.parent / 'masks'
        if not mask_dir.is_dir():
            continue
        images = {p.stem: p for p in image_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES}
        masks = {p.stem: p for p in mask_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES}
        shared = sorted(set(images) & set(masks))
        if shared:
            candidates.append((len(shared), image_dir, mask_dir, images, masks, shared))
    if not candidates:
        raise RuntimeError('Nisam pronašao sibling images/ i masks/ foldere u preuzetoj arhivi.')
    return max(candidates, key=lambda item: item[0])

pair_count, source_images_dir, source_masks_dir, source_images, source_masks, pair_ids = locate_pair_folders(extract_dir)
saved = sum(write_pair(item_id, source_images[item_id], source_masks[item_id]) for item_id in tqdm(pair_ids, desc='Preparing local V7 pairs'))
if saved != EXPECTED_PAIRS:
    raise RuntimeError(f'Očekivano je {EXPECTED_PAIRS} ispravnih lokalnih V7 parova, a pronađeno je {saved}. Proveri da li je uploadovan pravi colab_training_dataset.zip.')
print('Source images:', source_images_dir)
print('Source masks: ', source_masks_dir)
print(f'Prepared {saved} identical local V7 training pairs.')


In [ ]:
# 4) U-Net arhitektura, standardizacija i augmentacije 
SIZE = 512
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

class Block(nn.Module):
    def __init__(self,a,b):
        super().__init__(); self.x=nn.Sequential(nn.Conv2d(a,b,3,1,1),nn.BatchNorm2d(b),nn.ReLU(),nn.Conv2d(b,b,3,1,1),nn.BatchNorm2d(b),nn.ReLU())
    def forward(self,x): return self.x(x)

class UNet(nn.Module):
    def __init__(self):
        super().__init__(); self.p=nn.MaxPool2d(2)
        self.a,self.b,self.c,self.d,self.m=Block(1,32),Block(32,64),Block(64,128),Block(128,256),Block(256,512)
        self.u4,self.z4=nn.ConvTranspose2d(512,256,2,2),Block(512,256); self.u3,self.z3=nn.ConvTranspose2d(256,128,2,2),Block(256,128)
        self.u2,self.z2=nn.ConvTranspose2d(128,64,2,2),Block(128,64); self.u1,self.z1=nn.ConvTranspose2d(64,32,2,2),Block(64,32); self.o=nn.Conv2d(32,1,1)
    def forward(self,x):
        a=self.a(x); b=self.b(self.p(a)); c=self.c(self.p(b)); d=self.d(self.p(c)); m=self.m(self.p(d))
        d=self.z4(torch.cat((self.u4(m),d),1)); c=self.z3(torch.cat((self.u3(d),c),1)); b=self.z2(torch.cat((self.u2(c),b),1)); a=self.z1(torch.cat((self.u1(b),a),1)); return self.o(a)

def standardize_polarity(x):
    h,w=x.shape; y0,y1=round(.25*h),round(.70*h)
    lungs=np.concatenate((x[y0:y1,round(.16*w):round(.40*w)].ravel(),x[y0:y1,round(.60*w):round(.84*w)].ravel()))
    centre=x[round(.25*h):round(.75*h),round(.44*w):round(.56*w)]
    return cv2.bitwise_not(x) if lungs.size and np.median(lungs)>np.median(centre) else x

def norm(x):
    lo,hi=np.percentile(x,[1,99]); return np.clip((x.astype(np.float32)-lo)/max(hi-lo,1),0,1)

def paired_paths():
    out=[]
    for image_path in sorted(IMAGE_DIR.iterdir()):
        mask_path=MASK_DIR/f'{image_path.stem}.png'
        if image_path.suffix.lower() in IMAGE_SUFFIXES and mask_path.exists(): out.append((image_path,mask_path))
    if len(out)<20: raise RuntimeError('Potrebno je najmanje 20 uparenih slika i maski.')
    return out

class LungDataset(Dataset):
    def __init__(self,pairs,augment=False): self.pairs,self.augment=pairs,augment
    def __len__(self): return len(self.pairs)
    def __getitem__(self,index):
        image_path,mask_path=self.pairs[index]
        image=cv2.resize(cv2.imread(str(image_path),0),(SIZE,SIZE)); mask=cv2.resize(cv2.imread(str(mask_path),0),(SIZE,SIZE),interpolation=cv2.INTER_NEAREST)>127
        image=norm(standardize_polarity(image))
        if self.augment:
            if random.random()<.5: image,mask=image[:,::-1].copy(),mask[:,::-1].copy()
            image=np.clip(image*random.uniform(.7,1.35)+random.uniform(-.15,.15),0,1); image=np.power(image,random.uniform(.65,1.45))
            if random.random()<.5: image=np.clip(image+np.random.normal(0,.03,image.shape),0,1)
        return torch.from_numpy(image[None].astype(np.float32)),torch.from_numpy(mask[None].astype(np.float32))

def dice(logits,target,loss=False):
    prediction=torch.sigmoid(logits); prediction=prediction if loss else (prediction>=.5)
    return ((2*(prediction*target).sum((1,2,3))+1e-6)/(prediction.sum((1,2,3))+target.sum((1,2,3))+1e-6)).mean()

print('Device:',DEVICE)

In [ ]:
# 5) Fine-tuning. Najbolji checkpoint i istorija validacije se čuvaju u /content.
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
pairs=paired_paths(); random.shuffle(pairs); split=round(.85*len(pairs)); train_pairs,val_pairs=pairs[:split],pairs[split:]
model=UNet().to(DEVICE); optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE); criterion=nn.BCEWithLogitsLoss(); best_score=0.; history=[]
for epoch in range(1,EPOCHS+1):
    model.train(); losses=[]
    for image,mask in tqdm(DataLoader(LungDataset(train_pairs,True),BATCH_SIZE,shuffle=True),desc=f'Epoch {epoch}/{EPOCHS}'):
        image,mask=image.to(DEVICE),mask.to(DEVICE); logits=model(image); loss=criterion(logits,mask)+1-dice(logits,mask,True)
        optimizer.zero_grad(); loss.backward(); optimizer.step(); losses.append(loss.item())
    model.eval(); scores=[]
    with torch.inference_mode():
        for image,mask in DataLoader(LungDataset(val_pairs),BATCH_SIZE): scores.append(dice(model(image.to(DEVICE)),mask.to(DEVICE)).item())
    score=float(np.mean(scores)); history.append({'epoch':epoch,'train_loss':float(np.mean(losses)),'validation_dice':score})
    print(f'Epoch {epoch}: validation Dice={score:.4f}')
    if score>best_score:
        best_score=score; torch.save({'state':model.state_dict(),'dice':best_score},MODEL_PATH); print('Saved best model.')
history_path=OUTPUT_DIR/'training_history.csv'; pd.DataFrame(history).to_csv(history_path,index=False)
checkpoint=torch.load(MODEL_PATH,map_location=DEVICE,weights_only=True); model.load_state_dict(checkpoint['state']); model.eval()
print(f'Finished. Best validation Dice: {checkpoint["dice"]:.4f}')

In [ ]:
# 6) Analiza treninga — minimizacija gubitka i validacioni Dice po epohama.
training_history = pd.read_csv(history_path)
display(training_history)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(training_history['epoch'], training_history['train_loss'], marker='o', color='tab:red')
ax[0].set(title='Training loss', xlabel='Epoch', ylabel='BCE + Dice loss'); ax[0].grid(alpha=.25)
ax[1].plot(training_history['epoch'], training_history['validation_dice'], marker='o', color='tab:blue')
ax[1].set(title='Validation Dice', xlabel='Epoch', ylabel='Dice'); ax[1].set_ylim(0, 1); ax[1].grid(alpha=.25)
fig.tight_layout(); plt.show()
print(f'Best validation Dice: {training_history.validation_dice.max():.4f}')
print('Training history CSV:', history_path)

In [ ]:
# 7) Ručni upload TEST sekvence: MP4, jedna PNG/JPG slika ili ZIP PNG sekvenca.
from google.colab import files
uploaded=files.upload()
if not uploaded: raise RuntimeError('Nije izabran test fajl.')
test_name=next(iter(uploaded)); test_path=WORKDIR/test_name; shutil.move(test_name,test_path)

def load_test(path):
    if path.suffix.lower()=='.zip':
        folder=WORKDIR/f'{path.stem}_frames'; folder.mkdir(exist_ok=True)
        with zipfile.ZipFile(path) as archive: archive.extractall(folder)
        frames=[cv2.imread(str(p),0) for p in sorted(folder.rglob('*')) if p.suffix.lower() in IMAGE_SUFFIXES]; frames=[x for x in frames if x is not None]
        return frames,None,path.stem
    if path.suffix.lower() in IMAGE_SUFFIXES:
        frame=cv2.imread(str(path),0); return [frame],None,path.stem
    cap=cv2.VideoCapture(str(path)); fps=float(cap.get(cv2.CAP_PROP_FPS)) or 12.; frames=[]
    while True:
        ok,frame=cap.read()
        if not ok: break
        frames.append(cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY))
    cap.release(); return frames,fps,path.stem

raw_frames,fps,sequence_name=load_test(test_path)
if not raw_frames: raise RuntimeError('Test fajl nema čitljive frejmove.')
print(f'Loaded {len(raw_frames)} frame(s).')

In [ ]:
# 8) Sekvencijalna normalizacija, odvojena segmentacija oba krila, grafik i merenja.
def normalize_sequence(frames):
    canonical=[standardize_polarity(x) for x in frames]; stats=[]
    for x in canonical:
        h,w=x.shape; dy=max(1,round(.05*h)); dx=max(1,round(.05*w)); stats.append(np.percentile(x[dy:h-dy,dx:w-dx],(1,99)))
    stats=np.asarray(stats,np.float32); window=min(5,len(stats) if len(stats)%2 else len(stats)-1)
    if window>=3: stats=ndimage.median_filter(stats,size=(window,1),mode='nearest')
    return [np.clip((x.astype(np.float32)-low)*255/max(high-low,1),0,255).astype(np.uint8) for x,(low,high) in zip(canonical,stats)]

def infer(mask_input):
    h,w=mask_input.shape; tensor=torch.from_numpy((cv2.resize(mask_input,(SIZE,SIZE)).astype(np.float32)/255)[None,None]).to(DEVICE)
    with torch.inference_mode(): mask=torch.sigmoid(model(tensor))[0,0].cpu().numpy()>=.5
    labels,n=ndimage.label(mask)
    if n: mask=np.isin(labels,np.argsort(np.asarray(ndimage.sum(mask,labels,range(1,n+1))))[-min(2,n):]+1)
    return cv2.resize(mask.astype(np.uint8),(w,h),interpolation=cv2.INTER_NEAREST).astype(bool)

def split(mask):
    labels,n=ndimage.label(mask); parts=[labels==i for i in range(1,n+1)]
    if len(parts)>=2: parts=sorted(parts,key=lambda item:np.nonzero(item)[1].mean()); return parts[0],parts[-1]
    if len(parts)==1:
        _,xs=np.nonzero(parts[0]); xx=np.arange(mask.shape[1])[None,:]; return parts[0]&(xx<=np.median(xs)),parts[0]&(xx>np.median(xs))
    return np.zeros_like(mask,bool),np.zeros_like(mask,bool)

def measure(mask):
    y,_=np.nonzero(mask)
    if not len(y): return (np.nan,np.nan,np.nan,0)
    apex=float(y[y<=y.min()+2].mean()); bottom=float(np.quantile(y,.98)); return apex,bottom,bottom-apex,int(mask.sum())

frames=normalize_sequence(raw_frames); rows=[]
for i,frame in enumerate(tqdm(frames,desc='Segmenting sequence')):
    left,right=split(infer(frame)); la,lb,lh,larea=measure(left); ra,rb,rh,rarea=measure(right)
    rows.append({'frame':i,'left_apex_y':la,'left_bottom_y':lb,'left_height_px':lh,'left_area_px':larea,'right_apex_y':ra,'right_bottom_y':rb,'right_height_px':rh,'right_area_px':rarea})
results=pd.DataFrame(rows); csv_path=OUTPUT_DIR/f'{sequence_name}_measurements.csv'; results.to_csv(csv_path,index=False)
time=results.frame/(fps or 12.); window=min(7,len(results) if len(results)%2 else len(results)-1); smooth=lambda x:savgol_filter(x,window,2) if window>=3 else x
fig,ax=plt.subplots(2,1,figsize=(12,8),sharex=True)
for side,color in (('left','tab:blue'),('right','tab:green')):
    ax[0].plot(time,smooth(results[f'{side}_height_px']),color=color,lw=2,label=f'{side.title()} lung'); ax[1].plot(time,results[f'{side}_area_px'],color=color,lw=2,label=f'{side.title()} lung')
ax[0].set_ylabel('Height [pixels]'); ax[0].legend(); ax[1].set(xlabel='Time [s]' if fps else 'Frame',ylabel='Area [pixels squared]'); ax[1].legend(); fig.tight_layout()
graph_path=OUTPUT_DIR/f'{sequence_name}_graphs.png'; fig.savefig(graph_path,dpi=180); plt.show(); display(results)
print('Model:',MODEL_PATH); print('Measurements:',csv_path); print('Graph:',graph_path)

In [ ]:
# 9) Segmentacioni video — prikazan direktno u Colab-u.
def render_frame(frame,left,right,index,fps):
    h,w=frame.shape; original=cv2.cvtColor(frame,cv2.COLOR_GRAY2BGR); overlay=original.copy(); binary=np.zeros_like(overlay); values=[]
    for mask,label,color in ((left,'LEFT',(255,180,0)),(right,'RIGHT',(0,220,0))):
        layer=np.full_like(overlay,color); overlay[mask]=cv2.addWeighted(overlay[mask],.5,layer[mask],.5,0); binary[mask]=color
        contour,_=cv2.findContours(mask.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE); cv2.drawContours(overlay,contour,-1,color,max(1,round(w/240)))
        apex,bottom,height,area=measure(mask); values.append((apex,bottom,height,area))
        if area:
            x0,x1=(0,w//2-1) if label=='LEFT' else (w//2,w-1)
            for yy,kind in ((round(apex),'APEX'),(round(bottom),'BASE')):
                cv2.line(overlay,(x0,yy),(x1,yy),color,max(1,round(w/320))); cv2.putText(overlay,f'{label} {kind}',(x0+6,max(20,yy-7)),cv2.FONT_HERSHEY_SIMPLEX,.43,color,1,cv2.LINE_AA)
    for panel,title in zip((original,overlay,binary),('STANDARDIZED INPUT','LEFT + RIGHT LUNG SEGMENTATION','SEPARATE BINARY MASKS')):
        cv2.rectangle(panel,(0,0),(w,31),(0,0,0),-1); cv2.putText(panel,title,(10,22),cv2.FONT_HERSHEY_SIMPLEX,.6,(255,255,255),1,cv2.LINE_AA)
    output=cv2.hconcat((original,overlay,binary)); lv,rv=values; seconds=index/(fps or 12.)
    caption=f'Frame {index} | {seconds:.2f} s | Left: H {lv[2]:.0f}px, A {lv[3]} | Right: H {rv[2]:.0f}px, A {rv[3]}'
    cv2.rectangle(output,(0,h-34),(output.shape[1],h),(0,0,0),-1); cv2.putText(output,caption,(10,h-11),cv2.FONT_HERSHEY_SIMPLEX,.65,(255,255,255),2,cv2.LINE_AA)
    return output

video_path=OUTPUT_DIR/f'{sequence_name}_segmentation.mp4'; writer=None
for index,frame in enumerate(tqdm(frames,desc='Rendering video')):
    left,right=split(infer(frame)); rendered=render_frame(frame,left,right,index,fps)
    if writer is None:
        writer=cv2.VideoWriter(str(video_path),cv2.VideoWriter_fourcc(*'mp4v'),fps or 12.,(rendered.shape[1],rendered.shape[0]))
        if not writer.isOpened(): raise RuntimeError('Ne mogu da napravim MP4 izlaz.')
    writer.write(rendered)
if writer: writer.release()
h264_path=OUTPUT_DIR/f'{sequence_name}_segmentation_h264.mp4'
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(video_path),'-vcodec','libx264','-pix_fmt','yuv420p',str(h264_path)],check=True)
display(Video(str(h264_path),embed=True,html_attributes='controls width=100%'))
print('Video:',h264_path)